# 45. 카드·순위 고정, 원문 근거 보완 비교

목적은 구조화 fact에 연결된 원문 근거를 기존 답변 입력에 보완했을 때의 차이 확인이다. 42번 질문 48개, 운영 benefit/parent-child 청크를 읽기 전용 재사용한다.

A는 저장된 S_THEN_U 근거·답변, B는 카드·순위·첫 근거와 5개/카드·64,000 UTF-8 bytes 예산을 유지한 재구성이다. 추가만 11쌍과 교체 포함 7쌍을 분리한다. 나머지 78쌍은 동일 답변 재사용이다.

이 노트북 전체의 네 코드 셀은 저장 자료의 검증·표시에만 사용한다. 유료 실행은 재개·실패 원응답·비용 기록이 필요한 `45_run_answers.py`가 맡았다. 원문 연결/패키지는 `45_prepare_packages.py`, 쌍대 평가는 `45_evaluate_results.py`, 최종 감사는 `45_completion_audit.py`에 분리했다. 새 운영 경로나 새 DB를 만들지 않는다.

API 재호출은 이 노트북에 없다. 독립 holdout/반복 평균/사람 블라인드/운영 채택 결과가 아니며 OCR 통과 가정과 불완전한 제목 계층은 그대로다.

In [1]:
from pathlib import Path
import importlib.util
import sys
import pandas as pd
from IPython.display import display
HERE = next(p / 'notebooks/scripts' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks/scripts/45_experiment_io.py').is_file())
spec = importlib.util.spec_from_file_location('io45_notebook', HERE / '45_experiment_io.py')
I = importlib.util.module_from_spec(spec); sys.modules[spec.name] = I; spec.loader.exec_module(I)
runner = I.load('runner45_notebook', '45_run_answers.py')
contract, prepared = runner.validate()
P = I.load('packages45_notebook', '45_prepare_packages.py')
display(P.validate_prepared(prepared))
P.validate_change_audit(prepared, I.read(I.OUT / 'package_change_audit.json'))
display(I.preserve())
print('96 packages, 192 logical rows; frozen code and input hashes valid')

{'validated_packages': 96, 'fixed_cards_and_payloads': True}

{'passed': True, 'checked_files': 25069, 'changed': []}

96 packages, 192 logical rows; frozen code and input hashes valid


## 원문 연결과 실제 변경 범위

연결 상태의 분모는 보완 대상으로 선택된 고유 (프로필, fact) 271개다. 좌표·원문 줄·발췌·관계 범위를 엄격하게 확인하며 좌표 수리를 하지 않는다. 제외는 OCR 실패나 의미 오류를 뜻하지 않는다. 검증된 48개가 DB 전체 3,403개에 대한 정확도 점수는 아니다. 실제 패키지 96쌍의 추가/교체/동일/연결 미확인/기존 복귀 유지도 구분한다.

In [2]:
summary = I.read(I.OUT / 'preparation_summary.json')
effects = I.read(I.OUT / 'package_change_audit.json')
display(summary['by_profile'])
display(pd.DataFrame(effects['records']).groupby(['profile', 'effect_type']).size().rename('pairs').reset_index())
assert sum(r['treatment_status'] == 'changed' for r in prepared['records']) == 18

{'card_page_section_benefit': {'treatment': {'same_payload': 10,
   'mapping_unavailable': 9,
   'fallback_unchanged': 18,
   'changed': 11},
  'mapping': {'invalid_source': 96, 'verified': 30, 'unmapped': 10}},
 'parent_child_bundle': {'treatment': {'mapping_unavailable': 20,
   'fallback_unchanged': 18,
   'changed': 7,
   'same_payload': 3},
  'mapping': {'invalid_source': 90, 'unmapped': 27, 'verified': 18}}}

,profile,effect_type,pairs
0,card_page_section_benefit,append_only,6
1,card_page_section_benefit,fallback_unchanged,18
2,card_page_section_benefit,mapping_unavailable,9
3,card_page_section_benefit,replacement,5
4,card_page_section_benefit,same_payload,10
5,parent_child_bundle,append_only,5
6,parent_child_bundle,fallback_unchanged,18
7,parent_child_bundle,mapping_unavailable,20
8,parent_child_bundle,replacement,2
9,parent_child_bundle,same_payload,3


## 자동 진단과 원문 쌍대 판정

자동 충족은 39번 문자열·수치 기반 규칙이며 원문 의미 판정과 다르다. 양성 질문 전체의 분모는 프로필·방식별 44이고 실패도 포함한다. 유형별 표는 각 유형 질문 수가 분모다. 원문 근거 회수는 선택 fact에 필요한 원문 줄의 포함 여부로 같은 프로필 A/B만 비교하는 진단이다.

쌍대 개선/퇴행/동률은 실제 변경 18쌍을 원문과 소유 인용으로 검토한다. 핵심 PASS와 전체 주장 정확성은 다르며 새 수치·조건·인용 오류 또는 중요한 조건 손실 한 건도 B 선정을 막는다. 전체 48쌍에는 동일 입력 답변 재사용을 동률로 더하며 독립 재생성·절대 수동 정확도라고 부르지 않는다.

In [3]:
automatic = I.read(I.OUT / 'automatic_evaluation.json')
manual = I.read(I.OUT / 'source_review_summary.json')
assert len(automatic['rows']) == 192 and automatic['baseline_automatic_scores_reproduced'] == 96
display(pd.DataFrame(automatic['metrics']))
display(pd.DataFrame(manual['summary']))
display(pd.DataFrame(manual['overall_paired']))
display(pd.DataFrame(manual['records'])[['query_id','profile','A_core','B_core','outcome','finding']])
print('개발 실험 선정:', manual['decision'])

,profile,group,mode,n,automatic_complete,success,unsupported_percent_claims,citation_ownership,fixed_scope,witness_hit_facts,witness_planned_facts
0,card_page_section_benefit,all,A,44,31,44,1,44,44,31,161
1,card_page_section_benefit,all,B,44,33,44,1,44,44,36,161
2,card_page_section_benefit,numeric,A,16,12,16,1,16,16,14,58
3,card_page_section_benefit,numeric,B,16,13,16,1,16,16,15,58
4,card_page_section_benefit,proper_noun,A,16,16,16,0,16,16,6,11
5,card_page_section_benefit,proper_noun,B,16,16,16,0,16,16,6,11
6,card_page_section_benefit,semantic,A,12,3,12,0,12,12,11,92
7,card_page_section_benefit,semantic,B,12,4,12,0,12,12,15,92
8,parent_child_bundle,all,A,44,30,44,0,44,44,10,160
9,parent_child_bundle,all,B,44,30,44,0,44,44,20,160


,profile,effect_type,n,win,loss,tie,uncertain
0,card_page_section_benefit,all_changed,11,4,2,5,0
1,card_page_section_benefit,append_only,6,1,1,4,0
2,card_page_section_benefit,replacement,5,3,1,1,0
3,parent_child_bundle,all_changed,7,2,3,2,0
4,parent_child_bundle,append_only,5,2,1,2,0
5,parent_child_bundle,replacement,2,0,2,0,0


,profile,n,win,loss,tie,uncertain
0,card_page_section_benefit,48,4,2,42,0
1,parent_child_bundle,48,2,3,43,0


,query_id,profile,A_core,B_core,outcome,finding
0,E08,card_page_section_benefit,PASS,PASS,tie,양쪽 모두 약국 5% 이상 할인 카드를 소유 근거로 입증했다. 왼쪽의 신한카드 추가...
1,E08,parent_child_bundle,PASS,PASS,tie,양쪽 모두 약국 5% 핵심요구를 충족한다. 왼쪽은 롯데 생활업종 한도를 실적별 3천...
2,E14,card_page_section_benefit,PASS,PASS,tie,왼쪽 My WE:SH와 오른쪽 taptap DIGITAL 모두 한 카드에서 GS25...
3,E22,card_page_section_benefit,PASS,PASS,win,"양쪽 모두 다이소 7%, 건당 5천 원, 오프라인·입점매장 제외·전월실적·통합한도를..."
4,E23,card_page_section_benefit,PASS,PASS,tie,핵심 쿠팡 적립 혜택은 양쪽 모두 충족한다. 왼쪽만 발급 후 와우 멤버십을 해지해도...
5,E30,card_page_section_benefit,PASS,PASS,win,"양쪽 모두 쿠팡 10%, 건당 2만 원, 공식 홈페이지·앱 결제와 실적별 영역 한도..."
6,E31,card_page_section_benefit,PASS,PASS,win,"양쪽 모두 기본 1.2%, 실적 50만 원·월 대상금액 300만 원·월 한도 3만6..."
7,E31,parent_child_bundle,PASS,PASS,loss,"양쪽 모두 기본 1.2%, 실적 50만 원, 월 대상금액 300만 원을 설명한다. ..."
8,E33,parent_child_bundle,PASS,PASS,win,양쪽 모두 커피 혜택을 제시한다. 왼쪽은 오른쪽에서 미확인이라고 했던 LOCA LI...
9,E34,card_page_section_benefit,PASS,PASS,tie,양쪽 모두 I-어디로든 그린카드의 지정 카페 10% 적립·월 5천 포인트와 가온의 ...


개발 실험 선정: {'card_page_section_benefit': 'A', 'parent_child_bundle': 'A'}


## 비용·자원과 결론 자료

비용은 신규 물리 API 응답의 실제 usage에 공식 단가를 적용한 값이며 청구서 실측이 아니다. 실패·재시도도 포함하며 로컬 78개 답변 재사용과 provider cached input을 구별한다. 시간·최대 RSS는 단일 답변 실행 프로세스 범위이고 Codex 검토 시간·플랫폼 요금은 미측정이다. 새 임베딩·질문 파서·검색·BGE는 0회다.

[결과 보고서](../docs/report/45_evidence_supplement_comparison_report.html) · [원문 검토](data/45_evidence_supplement_comparison/source_reviews.json) · [감사 기록](data/45_evidence_supplement_comparison/completion_audit.json)

향상/퇴행·채택/거절·한계·다음 검증은 HTML 마지막 요약과 질문별 표에 적는다. 현재 운영 코드는 수정하지 않는다.

In [4]:
cost = I.read(I.OUT / 'cost_accounting.json')
resources = I.read(I.OUT / 'answer_resources.json')
answers = I.read(I.OUT / 'answers.json')['records']
assert len(answers) == 96 and sum(r['origin'] == 'reused_A' for r in answers) == 78
assert sum(r['origin'] == 'changed_B' for r in answers) == 18
display(cost['summary']); display(resources)
print('기존 검색 카드·순위 고정. 운영 적용 여부:', automatic['production_adopted'])

{'record_count': 18,
 'known_priced_records': 18,
 'unknown_cost_records': 0,
 'duplicate_saved_responses': 0,
 'known_cost_usd': '0.026177165',
 'tokens': {'input_tokens': 138626,
  'output_tokens': 22574,
  'ordinary_input_tokens': 54,
  'cached_input_tokens': 21189,
  'cache_write_tokens': 117383,
  'embedding_tokens': 0},
 'billed_cost_usd': None}

{'wall_seconds_this_process': 327.3810807919999,
 'cpu_seconds_this_process': 14.41341036,
 'peak_rss_kib': 241808,
 'new_api_requests': 18,
 'new_embeddings': 0,
 'new_parser_calls': 0,
 'new_bge_calls': 0,
 'gpu': '사용 안함',
 'source': 'single sequential answer process; preparation and Codex review excluded'}

기존 검색 카드·순위 고정. 운영 적용 여부: False
